Financial Metrics with PyNance

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np

# -----------------------------------
# Download Historical Data
# -----------------------------------

ticker = "AAPL"

df = yf.download(
    ticker,
    start="2023-01-01"
)

# -----------------------------------
# RETURNS
# -----------------------------------

df["Returns"] = (
    df["Close"]
    .pct_change()
)

# -----------------------------------
# VOLATILITY
# -----------------------------------

annual_volatility = (
    df["Returns"].std()
    * np.sqrt(252)
)

# -----------------------------------
# SHARPE RATIO
# -----------------------------------

risk_free_rate = 0.02

sharpe_ratio = (
    (
        df["Returns"].mean() * 252
        - risk_free_rate
    )
    /
    annual_volatility
)

# -----------------------------------
# SORTINO RATIO
# -----------------------------------

downside_returns = df.loc[
    df["Returns"] < 0,
    "Returns"
]

downside_std = (
    downside_returns.std()
    * np.sqrt(252)
)

sortino_ratio = (
    (
        df["Returns"].mean() * 252
        - risk_free_rate
    )
    /
    downside_std
)

# -----------------------------------
# MAXIMUM DRAWDOWN
# -----------------------------------

cumulative_returns = (
    1 + df["Returns"]
).cumprod()

rolling_peak = (
    cumulative_returns.cummax()
)

drawdown = (
    cumulative_returns
    / rolling_peak
    - 1
)

max_drawdown = drawdown.min()

# -----------------------------------
# VALUE AT RISK (95%)
# -----------------------------------

var_95 = np.percentile(
    df["Returns"].dropna(),
    5
)

# -----------------------------------
# CONDITIONAL VaR (CVaR)
# -----------------------------------

cvar_95 = df["Returns"][
    df["Returns"] <= var_95
].mean()

# -----------------------------------
# BETA vs S&P 500
# -----------------------------------

market = yf.download(
    "^GSPC",
    start="2023-01-01"
)

market["Returns"] = (
    market["Close"]
    .pct_change()
)

combined = pd.concat(
    [
        df["Returns"],
        market["Returns"]
    ],
    axis=1
).dropna()

combined.columns = [
    "Stock",
    "Market"
]

covariance = np.cov(
    combined["Stock"],
    combined["Market"]
)[0][1]

market_variance = np.var(
    combined["Market"]
)

beta = covariance / market_variance

# -----------------------------------
# OUTPUT
# -----------------------------------

print("\n===== FINANCIAL METRICS =====")

print(f"Annual Volatility: {annual_volatility:.4f}")
print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Sortino Ratio: {sortino_ratio:.4f}")
print(f"Maximum Drawdown: {max_drawdown:.4f}")
print(f"VaR (95%): {var_95:.4f}")
print(f"CVaR (95%): {cvar_95:.4f}")
print(f"Beta vs S&P 500: {beta:.4f}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


===== FINANCIAL METRICS =====
Annual Volatility: 0.2549
Sharpe Ratio: 1.0916
Sortino Ratio: 1.5691
Maximum Drawdown: -0.3336
VaR (95%): -0.0239
CVaR (95%): -0.0353
Beta vs S&P 500: 1.1549
